# 06 -- Park Geometry Analysis (Contact Luck v0.4)

**Contact Luck Prototype v0.4**

Does replacing bare venue IDENTITY (Version 0.3) with physical wall GEOMETRY (wall distance/height as a function of spray angle) improve out-of-sample probability quality?

Four controlled variants, same 2021-2023 training rows, same untouched 2024 validation rows (2025 never touched):

- `baseline_v02` -- no venue, no geometry.
- `venue_only_v03_candidate` -- `baseline_v02` + `venue_id` (Version 0.3).
- `geometry_only_v04_candidate` -- `baseline_v02` + wall-geometry features, NO `venue_id`.
- `venue_plus_geometry_v04_candidate` -- `baseline_v02` + `venue_id` + wall-geometry features.

**Adoption rule: this notebook does NOT automatically prefer a geometry candidate.** See the recommendation section near the end -- it is a starting point for judgment (it does not even attempt to automate "is the model learning physically plausible effects", the task's 8th criterion), never a substitute for reading the tables below.

**Contact Luck v0.4 remains a contact-only research prototype.** It does not yet model weather, air density, roof status, exact ball trajectory, defensive positioning, defensive execution, or batter-runner advancement.

> **This repository is a research prototype, not a validated public baseball statistic.** See `README.md` and `CLAUDE.md` for full scope and limitations.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from mlb_luck_score.config import CLASS_ORDER, PROCESSED_DATA_DIR, TRAIN_SEASONS, VALIDATION_SEASONS
from mlb_luck_score.data.clean_batted_balls import _spray_angle_degrees, _SPRAY_SECTOR_EDGES
from mlb_luck_score.data.join_park_geometry import (
    build_geometry_join_report,
    join_park_geometry,
    GEOMETRY_STATUS_OK,
)
from mlb_luck_score.data.park_geometry import (
    PARK_GEOMETRY_CONFIGS,
    PARK_GEOMETRY_POINTS,
    STANDARD_ANGLES,
    TEMPORARY_OR_SPECIAL_VENUE_IDS,
    interpolate_wall_geometry,
    resolve_geometry_config,
)
from mlb_luck_score.models.compare_geometry_aware import (
    ALL_VARIANTS,
    GEOMETRY_CANDIDATE_VARIANTS,
    VARIANT_BASELINE_V02,
    VARIANT_GEOMETRY_ONLY_V04_CANDIDATE,
    VARIANT_PARK_AWARE_V03_CANDIDATE,
    VARIANT_VENUE_PLUS_GEOMETRY_V04_CANDIDATE,
    compute_paired_bootstrap,
    recommend_geometry_adoption,
    run_geometry_aware_comparison,
)
from mlb_luck_score.models.train_contact_model import predict_proba_ordered, train_model

pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 60)

GEOMETRY_JOINED_PATH = PROCESSED_DATA_DIR / "cleaned_development_data_with_geometry.parquet"
VENUE_JOINED_PATH = PROCESSED_DATA_DIR / "cleaned_development_data_with_venue.parquet"

if GEOMETRY_JOINED_PATH.exists():
    df = pd.read_parquet(GEOMETRY_JOINED_PATH)
    print(f"Loaded pre-joined geometry data: {len(df)} rows from {GEOMETRY_JOINED_PATH}")
elif VENUE_JOINED_PATH.exists():
    venue_df = pd.read_parquet(VENUE_JOINED_PATH)
    df = join_park_geometry(venue_df)
    print(f"Joined geometry on the fly: {len(df)} rows")
else:
    df = None
    print(
        "No venue-joined data found. Run `make download-development-data && "
        "make clean-development-data && make download-game-metadata && "
        "make join-venue-metadata && make join-park-geometry` first, then re-run this notebook."
    )

Loaded pre-joined geometry data: 494173 rows from /Users/arihantaneja/Downloads/TrueLuckMLBStat/data/processed/cleaned_development_data_with_geometry.parquet


## 1. Geometry source and schema overview

See `mlb_luck_score.data.park_geometry` module docstring for the full sourcing methodology. **Every record's `review_status` is `agent_sourced_pending_human_review`** -- gathered by an AI coding agent from cited public sources (mostly individual-ballpark Wikipedia articles), not yet confirmed by a human maintainer. Treat this table as a documented Version 0.4 research placeholder.

In [2]:
print(f"Reviewed wall points:        {len(PARK_GEOMETRY_POINTS)}")
print(f"Geometry configurations:     {len(PARK_GEOMETRY_CONFIGS)}")
venues = sorted({c.venue_id for c in PARK_GEOMETRY_CONFIGS})
print(f"Venues covered:              {len(venues)}")
print(f"Temporary/special venues (deliberately NOT covered): {len(TEMPORARY_OR_SPECIAL_VENUE_IDS)}")
review_statuses = sorted({p.review_status for p in PARK_GEOMETRY_POINTS})
print(f"Review status(es) present:   {review_statuses}")
print(f"\nStandard angle grid (degrees): {STANDARD_ANGLES}")
print("0 = straightaway center field, negative = third-base/left-field side, positive = first-base/right-field side.")

Reviewed wall points:        165
Geometry configurations:     33
Venues covered:              30
Temporary/special venues (deliberately NOT covered): 7
Review status(es) present:   ['agent_sourced_pending_human_review']

Standard angle grid (degrees): (-45.0, -22.5, 0.0, 22.5, 45.0)
0 = straightaway center field, negative = third-base/left-field side, positive = first-base/right-field side.


## 2. Configuration counts by venue and effective dates

Three venues have more than one configuration because their real outfield walls changed during 2021-2024 (see each configuration's `notes` for citations).

In [3]:
config_rows = []
for c in sorted(PARK_GEOMETRY_CONFIGS, key=lambda c: (c.venue_name, c.effective_start_date)):
    config_rows.append({
        "venue_id": c.venue_id,
        "venue_name": c.venue_name,
        "geometry_config_id": c.geometry_config_id,
        "effective_start_date": c.effective_start_date,
        "effective_end_date": c.effective_end_date or "(current)",
        "n_points": len(c.points),
    })
config_df = pd.DataFrame(config_rows)
display(config_df)

multi_config_venues = config_df.groupby("venue_id").size()
multi_config_venues = multi_config_venues[multi_config_venues > 1]
print(f"\nVenues with more than one configuration: {list(multi_config_venues.index)} (Camden Yards, Rogers Centre, Comerica Park)")

,venue_id,venue_name,geometry_config_id,effective_start_date,effective_end_date,n_points
0,32,American Family Field,american_family_field_v1,2001-01-01,(current),5
1,1,Angel Stadium,angel_stadium_v1,2018-01-01,(current),5
2,2889,Busch Stadium,busch_stadium_v1,2006-01-01,(current),5
3,15,Chase Field,chase_field_v1,1998-01-01,(current),5
4,3289,Citi Field,citi_field_v1,2012-01-01,(current),5
5,2681,Citizens Bank Park,citizens_bank_park_v1,2004-01-01,(current),5
6,2394,Comerica Park,comerica_park_pre2023,2003-01-01,2023-03-29,5
7,2394,Comerica Park,comerica_park_2023_2024,2023-03-30,(current),5
8,19,Coors Field,coors_field_v1,2016-01-01,(current),5
9,22,Dodger Stadium,dodger_stadium_v1,1969-01-01,(current),5



Venues with more than one configuration: [2, 14, 2394] (Camden Yards, Rogers Centre, Comerica Park)


## 3. Geometry coverage by season, venue, and batted-ball type

In [4]:
if df is not None:
    report = build_geometry_join_report(df)
    print(f"Total rows:              {report.total_rows}")
    print(f"Rows with geometry:      {report.rows_with_geometry}")
    print(f"Coverage rate:           {report.coverage_rate:.4%}")
    print(f"\nCounts by geometry_status:")
    for status, count in sorted(report.counts_by_status.items(), key=lambda kv: -kv[1]):
        print(f"  {status:35s} {count:>8d}")
    print(f"\nConfigurations actually used in the data: {report.n_configurations_used}")
    print(f"Temporary/special-venue rows: {report.n_temporary_or_special_venue_rows}")
else:
    report = None
    print("Skipped -- no data loaded.")

Total rows:              494173
Rows with geometry:      443785
Coverage rate:           89.8036%

Counts by geometry_status:
  ok                                    443785
  outside_modeled_angular_range          47012
  temporary_or_special_venue              3147
  missing_spray_angle                      229

Configurations actually used in the data: 33
Temporary/special-venue rows: 3147


In [5]:
if df is not None:
    print("Coverage by season:")
    for season, n in sorted(report.counts_by_season.items()):
        total = int((df["season"].astype(str) == str(season)).sum())
        rate = n / total if total else float("nan")
        print(f"  {season}: {n}/{total} ({rate:.2%})")

    print("\nCoverage by batted-ball type:")
    for bb_type, n in sorted(report.counts_by_bb_type.items(), key=lambda kv: -kv[1]):
        total = int((df["bb_type"].astype(str) == str(bb_type)).sum())
        rate = n / total if total else float("nan")
        print(f"  {str(bb_type):15s} {n:>7d}/{total:<7d} ({rate:.2%})")
else:
    print("Skipped -- no data loaded.")

Coverage by season:
  2021: 107036/121699 (87.95%)
  2022: 112333/124261 (90.40%)


  2023: 112272/124233 (90.37%)


  2024: 112144/123980 (90.45%)

Coverage by batted-ball type:
  ground_ball      191966/214032  (89.69%)
  fly_ball         120767/128383  (94.07%)
  line_drive       110375/117290  (94.10%)
  popup             20677/34458   (60.01%)
  nan                   0/0       (nan%)


## 4. Wall-profile visuals (one park per subplot)

Spray angle (degrees, x-axis) vs. wall distance (feet, y-axis) for a sample of parks -- including both eras of the two most dimensionally-distinctive mid-window renovations. Plotted separately per park, not combined into one chart.

In [6]:
if df is not None:
    sample_config_ids = [
        "fenway_park_v1", "coors_field_v1", "yankee_stadium_v1", "oracle_park_v1",
        "camden_yards_pre2022", "camden_yards_2022_2024",
        "rogers_centre_pre2023", "rogers_centre_2023_2024",
        "petco_park_v1",
    ]
    configs_by_id = {c.geometry_config_id: c for c in PARK_GEOMETRY_CONFIGS}

    fig, axes = plt.subplots(3, 3, figsize=(13, 11))
    for ax, config_id in zip(axes.ravel(), sample_config_ids):
        cfg = configs_by_id[config_id]
        angles = [p.spray_angle_degrees for p in cfg.points]
        distances = [p.wall_distance_feet for p in cfg.points]
        ax.plot(angles, distances, marker="o")
        ax.set_title(f"{cfg.venue_name}\n({config_id})", fontsize=9)
        ax.set_xlabel("Spray angle (deg)")
        ax.set_ylabel("Wall distance (ft)")
        ax.set_ylim(280, 430)
    plt.tight_layout()
    plt.show()
else:
    print("Skipped -- no data loaded.")

/var/folders/k_/c1x6qgmx0b56khm3k49fgfq80000gn/T/ipykernel_67410/1412906698.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Verification of spray-angle orientation

Confirms `park_geometry`'s standard angle grid uses the SAME sign convention as `clean_batted_balls.spray_angle_approx`: 0 = center, negative = left/third-base side, positive = right/first-base side.

In [7]:
left_angle = _spray_angle_degrees(hc_x=90.0, hc_y=150.0)
center_angle = _spray_angle_degrees(hc_x=125.42, hc_y=100.0)
right_angle = _spray_angle_degrees(hc_x=170.0, hc_y=150.0)
print(f"Synthetic left-field hit:   hc_x=90.0  -> spray_angle_approx={left_angle:.2f} deg (negative = left, as expected)")
print(f"Synthetic center-field hit: hc_x=125.42 -> spray_angle_approx={center_angle:.2f} deg (~0 = center, as expected)")
print(f"Synthetic right-field hit:  hc_x=170.0 -> spray_angle_approx={right_angle:.2f} deg (positive = right, as expected)")
print(f"\nSTANDARD_ANGLES boundary: {STANDARD_ANGLES[0]} / {STANDARD_ANGLES[-1]}")
print(f"clean_batted_balls fair-territory sector boundary: {_SPRAY_SECTOR_EDGES[0]} / {_SPRAY_SECTOR_EDGES[-1]}")
assert STANDARD_ANGLES[0] == _SPRAY_SECTOR_EDGES[0] and STANDARD_ANGLES[-1] == _SPRAY_SECTOR_EDGES[-1]
print("Boundaries match -- geometry interpolation is never attempted outside the angular range already treated as modeled fair territory elsewhere in this repo.")

Synthetic left-field hit:   hc_x=90.0  -> spray_angle_approx=-36.27 deg (negative = left, as expected)
Synthetic center-field hit: hc_x=125.42 -> spray_angle_approx=0.00 deg (~0 = center, as expected)
Synthetic right-field hit:  hc_x=170.0 -> spray_angle_approx=42.72 deg (positive = right, as expected)

STANDARD_ANGLES boundary: -45.0 / 45.0
clean_batted_balls fair-territory sector boundary: -45.0 / 45.0
Boundaries match -- geometry interpolation is never attempted outside the angular range already treated as modeled fair territory elsewhere in this repo.


## 6. Wall-distance interpolation examples

Fenway Park: exact reviewed points are `measured`; any other angle within +-45 degrees is piecewise-linearly `interpolated` between its two nearest reviewed points.

In [8]:
fenway = resolve_geometry_config(3, "2022-06-01")
for angle in (-45.0, -35.0, -22.5, -10.0, 0.0, 10.0, 22.5, 35.0, 45.0):
    result = interpolate_wall_geometry(fenway, angle)
    print(
        f"angle={angle:>6.1f} deg -> distance={result.wall_distance_feet:>6.1f} ft "
        f"({result.distance_source_type:11s}, nearest_point_distance={result.nearest_point_distance_degrees:.2f} deg, "
        f"segment={result.segment_label})"
    )

angle= -45.0 deg -> distance= 310.0 ft (measured   , nearest_point_distance=0.00 deg, segment=left_field_line)
angle= -35.0 deg -> distance= 340.7 ft (interpolated, nearest_point_distance=10.00 deg, segment=between left_field_line and left_center (nearest: left_field_line))
angle= -22.5 deg -> distance= 379.0 ft (measured   , nearest_point_distance=0.00 deg, segment=left_center)
angle= -10.0 deg -> distance= 385.1 ft (interpolated, nearest_point_distance=10.00 deg, segment=between left_center and center_field (nearest: center_field))
angle=   0.0 deg -> distance= 390.0 ft (measured   , nearest_point_distance=0.00 deg, segment=center_field)
angle=  10.0 deg -> distance= 385.6 ft (interpolated, nearest_point_distance=10.00 deg, segment=between center_field and right_center (nearest: center_field))
angle=  22.5 deg -> distance= 380.0 ft (measured   , nearest_point_distance=0.00 deg, segment=right_center)
angle=  35.0 deg -> distance= 336.7 ft (interpolated, nearest_point_distance=10.00 de

## 7. Distribution of projected distance-to-wall margins

In [9]:
if df is not None:
    has_margin = df["projected_distance_to_wall_margin"].notna()
    margins = df.loc[has_margin, "projected_distance_to_wall_margin"]
    print(margins.describe())

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(margins.clip(-300, 100), bins=80)
    ax.axvline(0, color="red", linestyle="--", label="wall (margin=0)")
    ax.set_xlabel("projected_distance_to_wall_margin (ft, clipped to [-300, 100] for display)")
    ax.set_ylabel("count")
    ax.set_title("Distribution of hit_distance_sc - wall_distance_in_spray_direction")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Skipped -- no data loaded.")

count    442677.000000
mean       -201.941049
std         136.661367
min        -417.604402
25%        -339.431867
50%        -202.682697
75%         -77.520232
max         131.523649
Name: projected_distance_to_wall_margin, dtype: float64


/var/folders/k_/c1x6qgmx0b56khm3k49fgfq80000gn/T/ipykernel_67410/3425560756.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Counts of near-wall plays

In [10]:
if df is not None:
    geo_rows = df[df["has_park_geometry"]]
    for col in ("near_wall_5ft", "near_wall_10ft", "near_wall_20ft", "projected_beyond_wall"):
        counts = geo_rows[col].value_counts(dropna=False)
        print(f"{col}: {dict(counts)}")
    print(f"\nhigh_wall_indicator: {dict(geo_rows['high_wall_indicator'].value_counts(dropna=False))}")
    print(f"temporary_or_special_venue rows: {int(df['temporary_or_special_venue'].sum())}")
else:
    print("Skipped -- no data loaded.")

near_wall_5ft: {np.False_: np.int64(434768), np.True_: np.int64(7909), <NA>: np.int64(1108)}
near_wall_10ft: {np.False_: np.int64(426766), np.True_: np.int64(15911), <NA>: np.int64(1108)}
near_wall_20ft: {np.False_: np.int64(411085), np.True_: np.int64(31592), <NA>: np.int64(1108)}
projected_beyond_wall: {np.False_: np.int64(414956), np.True_: np.int64(27721), <NA>: np.int64(1108)}

high_wall_indicator: {<NA>: np.int64(389368), np.False_: np.int64(48988), np.True_: np.int64(5429)}
temporary_or_special_venue rows: 3147


## 9. Four-way model comparison

Trains all four variants on IDENTICAL 2021-2023 rows and evaluates on the SAME untouched 2024 validation rows.

In [11]:
if df is not None:
    comparison, trained_models, proba_by_variant = run_geometry_aware_comparison(df)
    summary_rows = []
    for variant in ALL_VARIANTS:
        s = comparison[variant]
        summary_rows.append({
            "variant": variant,
            "multiclass_log_loss": s["multiclass_log_loss"],
            "expected_calibration_error": s["expected_calibration_error"],
            "home_run_ece": s["home_run_ece"],
            "argmax_accuracy (secondary)": s["argmax_accuracy_secondary"],
            "n": s["sample_count"],
        })
    display(pd.DataFrame(summary_rows).set_index("variant"))
    print("\nLog loss and ECE: LOWER is better. Accuracy is a SECONDARY number only.")
else:
    comparison, trained_models, proba_by_variant = None, None, None
    print("Skipped -- no data loaded.")

6 of 41 calibration bin(s) have fewer than 20 samples and are marked unreliable.


3 of 36 calibration bin(s) have fewer than 20 samples and are marked unreliable.


3 of 35 calibration bin(s) have fewer than 20 samples and are marked unreliable.


3 of 36 calibration bin(s) have fewer than 20 samples and are marked unreliable.


2 of 35 calibration bin(s) have fewer than 20 samples and are marked unreliable.


...

3 of 42 calibration bin(s) have fewer than 20 samples and are marked unreliable.


1 of 42 calibration bin(s) have fewer than 20 samples and are marked unreliable.


5 of 46 calibration bin(s) have fewer than 20 samples and are marked unreliable.


,multiclass_log_loss,expected_calibration_error,home_run_ece,argmax_accuracy (secondary),n
variant,,,,,
baseline_v02,0.670321,0.014413,0.002706,0.751457,122132
park_aware_v03_candidate,0.668930,0.014167,0.002977,0.750876,122132
geometry_only_v04_candidate,0.563283,0.016284,0.001019,0.777921,122132
venue_plus_geometry_v04_candidate,0.559193,0.016177,0.001074,0.779214,122132



Log loss and ECE: LOWER is better. Accuracy is a SECONDARY number only.


## 10. Calibration by outcome class

In [12]:
if comparison is not None:
    class_rows = []
    for variant in ALL_VARIANTS:
        row = {"variant": variant}
        row.update(comparison[variant]["expected_calibration_error_by_class"])
        class_rows.append(row)
    display(pd.DataFrame(class_rows).set_index("variant")[list(CLASS_ORDER)])
else:
    print("Skipped -- no comparison available.")

,out,single,double,triple,home_run
variant,,,,,
baseline_v02,0.031353,0.033749,0.004039,0.000218,0.002706
park_aware_v03_candidate,0.030179,0.034093,0.003362,0.000223,0.002977
geometry_only_v04_candidate,0.036916,0.040808,0.002481,0.000195,0.001019
venue_plus_geometry_v04_candidate,0.036841,0.040189,0.002531,0.000251,0.001074


## 11. Calibration by venue

In [13]:
if comparison is not None:
    base_venue = pd.DataFrame(comparison[VARIANT_BASELINE_V02]["calibration_by_venue"])
    for candidate in GEOMETRY_CANDIDATE_VARIANTS:
        cand_venue = pd.DataFrame(comparison[candidate]["calibration_by_venue"])
        by_venue = base_venue.merge(cand_venue, on="venue_id", suffixes=("_baseline", "_candidate"))
        by_venue["ece_delta"] = by_venue["ece_overall_candidate"] - by_venue["ece_overall_baseline"]
        by_venue = by_venue.sort_values("ece_delta")
        print(f"\n=== {candidate} vs baseline_v02 (most-improved venues first) ===")
        display(by_venue[["venue_id", "sample_count_baseline", "ece_overall_baseline", "ece_overall_candidate", "ece_delta", "reliable_baseline"]].head(10))
        print("... largest regressions:")
        display(by_venue[["venue_id", "sample_count_baseline", "ece_overall_baseline", "ece_overall_candidate", "ece_delta", "reliable_baseline"]].tail(5))
else:
    print("Skipped -- no comparison available.")


=== geometry_only_v04_candidate vs baseline_v02 (most-improved venues first) ===


,venue_id,sample_count_baseline,ece_overall_baseline,ece_overall_candidate,ece_delta,reliable_baseline
31,5381,103,0.070763,0.046884,-0.023879,True
27,32,3868,0.020408,0.015918,-0.004490,True
23,2394,3981,0.023499,0.020284,-0.003216,True
18,1,4010,0.020433,0.017830,-0.002604,True
19,2602,4004,0.017565,0.014991,-0.002574,True
21,22,3997,0.018880,0.017305,-0.001575,True
26,3289,3880,0.020612,0.020148,-0.000464,True
17,5325,4011,0.020717,0.020302,-0.000415,True
6,14,4164,0.018632,0.018265,-0.000367,True
25,17,3940,0.019180,0.018988,-0.000193,True


... largest regressions:


,venue_id,sample_count_baseline,ece_overall_baseline,ece_overall_candidate,ece_delta,reliable_baseline
28,4705,3824,0.015205,0.022274,0.007070,True
13,3,4103,0.016032,0.023706,0.007674,True
1,4169,4304,0.013983,0.021972,0.007989,True
30,5340,109,0.038569,0.067957,0.029388,True
32,2735,42,0.066350,0.124686,0.058337,False



=== venue_plus_geometry_v04_candidate vs baseline_v02 (most-improved venues first) ===


,venue_id,sample_count_baseline,ece_overall_baseline,ece_overall_candidate,ece_delta,reliable_baseline
31,5381,103,0.070763,0.054352,-0.016410,True
27,32,3868,0.020408,0.015190,-0.005218,True
19,2602,4004,0.017565,0.013427,-0.004138,True
23,2394,3981,0.023499,0.019982,-0.003517,True
18,1,4010,0.020433,0.017049,-0.003385,True
21,22,3997,0.018880,0.015759,-0.003121,True
17,5325,4011,0.020717,0.018298,-0.002419,True
24,12,3957,0.019714,0.017744,-0.001971,True
26,3289,3880,0.020612,0.019475,-0.001137,True
9,4,4122,0.016206,0.015118,-0.001087,True


... largest regressions:


,venue_id,sample_count_baseline,ece_overall_baseline,ece_overall_candidate,ece_delta,reliable_baseline
1,4169,4304,0.013983,0.020186,0.006203,True
13,3,4103,0.016032,0.022811,0.006778,True
33,3949,32,0.095968,0.115329,0.019361,False
30,5340,109,0.038569,0.076984,0.038414,True
32,2735,42,0.066350,0.137200,0.070850,False


## 12. Near-wall subgroup calibration

In [14]:
if comparison is not None:
    subgroup_rows = []
    for variant in ALL_VARIANTS:
        for label, s in comparison[variant]["geometry_subgroups"].items():
            subgroup_rows.append({
                "variant": variant, "subgroup": label,
                "sample_count": s["sample_count"], "ece": s["ece"], "log_loss": s["log_loss"],
            })
    subgroup_df = pd.DataFrame(subgroup_rows)
    for label in ("near_wall_5ft", "near_wall_10ft", "near_wall_20ft", "projected_beyond_wall", "geometry_available", "geometry_unavailable"):
        print(f"\n--- {label} ---")
        display(subgroup_df[subgroup_df["subgroup"] == label].set_index("variant")[["sample_count", "ece", "log_loss"]])
else:
    print("Skipped -- no comparison available.")


--- near_wall_5ft ---


,sample_count,ece,log_loss
variant,,,
baseline_v02,1952,0.118093,1.417421
park_aware_v03_candidate,1952,0.113734,1.391274
geometry_only_v04_candidate,1952,0.032280,1.104293
venue_plus_geometry_v04_candidate,1952,0.029188,1.039138



--- near_wall_10ft ---


,sample_count,ece,log_loss
variant,,,
baseline_v02,3989,0.104664,1.345986
park_aware_v03_candidate,3989,0.101402,1.322076
geometry_only_v04_candidate,3989,0.024361,1.024591
venue_plus_geometry_v04_candidate,3989,0.022069,0.961483



--- near_wall_20ft ---


...

,sample_count,ece,log_loss
variant,,,
baseline_v02,6788,0.080550,0.941676
park_aware_v03_candidate,6788,0.079649,0.920503
geometry_only_v04_candidate,6788,0.011775,0.577866
venue_plus_geometry_v04_candidate,6788,0.011335,0.525943



--- geometry_available ---


,sample_count,ece,log_loss
variant,,,
baseline_v02,110489,0.018917,0.654355
park_aware_v03_candidate,110489,0.018572,0.652656
geometry_only_v04_candidate,110489,0.017162,0.562137
venue_plus_geometry_v04_candidate,110489,0.016905,0.557074



--- geometry_unavailable ---


,sample_count,ece,log_loss
variant,,,
baseline_v02,11643,0.065309,0.821840
park_aware_v03_candidate,11643,0.065369,0.823373
geometry_only_v04_candidate,11643,0.022566,0.574160
venue_plus_geometry_v04_candidate,11643,0.022170,0.579308


## 13. Paired-bootstrap intervals

Game_pk-level paired bootstrap (see `compute_paired_bootstrap`): resamples GAMES with replacement so within-game correlation between plays is preserved; both the baseline and each candidate are evaluated on the SAME resampled rows each replicate.

In [15]:
if comparison is not None:
    training_eligible = df[df["eligible_for_training"].astype(bool)]
    val_df = training_eligible[training_eligible["season"].isin(VALIDATION_SEASONS)]
    from mlb_luck_score.features.build_contact_features import add_geometry_interaction_features
    val_df = add_geometry_interaction_features(val_df)
    y_true = val_df["outcome_class"].astype(str)

    bootstrap_by_candidate = {}
    for candidate in GEOMETRY_CANDIDATE_VARIANTS:
        bootstrap_by_candidate[candidate] = compute_paired_bootstrap(
            y_true, proba_by_variant[VARIANT_BASELINE_V02], proba_by_variant[candidate], val_df["game_pk"],
        )
        print(f"\n=== {candidate} vs baseline_v02 (n_reps={bootstrap_by_candidate[candidate]['log_loss']['n_reps']}, seed={bootstrap_by_candidate[candidate]['log_loss']['seed']}) ===")
        for metric, result in bootstrap_by_candidate[candidate].items():
            print(f"  {metric:15s} point_estimate={result['point_estimate']:+.6f}  95% CI=[{result['ci_low']:+.6f}, {result['ci_high']:+.6f}]")
else:
    bootstrap_by_candidate = None
    print("Skipped -- no comparison available.")


=== geometry_only_v04_candidate vs baseline_v02 (n_reps=500, seed=42) ===
  log_loss        point_estimate=-0.107038  95% CI=[-0.109402, -0.104200]
  ece             point_estimate=+0.001871  95% CI=[+0.000946, +0.002946]
  home_run_ece    point_estimate=-0.001687  95% CI=[-0.002466, -0.000940]
  brier_out       point_estimate=-0.020458  95% CI=[-0.021047, -0.019792]
  brier_single    point_estimate=-0.012208  95% CI=[-0.012670, -0.011765]
  brier_double    point_estimate=-0.011551  95% CI=[-0.012093, -0.011041]
  brier_triple    point_estimate=-0.000119  95% CI=[-0.000146, -0.000093]
  brier_home_run  point_estimate=-0.010124  95% CI=[-0.010619, -0.009721]



=== venue_plus_geometry_v04_candidate vs baseline_v02 (n_reps=500, seed=42) ===
  log_loss        point_estimate=-0.111128  95% CI=[-0.113603, -0.108238]
  ece             point_estimate=+0.001764  95% CI=[+0.000793, +0.002847]
  home_run_ece    point_estimate=-0.001632  95% CI=[-0.002524, -0.000882]
  brier_out       point_estimate=-0.021010  95% CI=[-0.021614, -0.020338]
  brier_single    point_estimate=-0.012274  95% CI=[-0.012734, -0.011813]
  brier_double    point_estimate=-0.011929  95% CI=[-0.012461, -0.011394]
  brier_triple    point_estimate=-0.000140  95% CI=[-0.000171, -0.000108]
  brier_home_run  point_estimate=-0.011196  95% CI=[-0.011710, -0.010753]


## 14. Example plays whose probabilities changed most

In [16]:
if comparison is not None:
    l1_diff = (proba_by_variant[VARIANT_BASELINE_V02][list(CLASS_ORDER)] - proba_by_variant[VARIANT_VENUE_PLUS_GEOMETRY_V04_CANDIDATE][list(CLASS_ORDER)]).abs().sum(axis=1)
    top_changed = l1_diff.sort_values(ascending=False).head(5)
    for idx in top_changed.index:
        row = val_df.loc[idx]
        print(f"\n--- event_id={row.get('event_id', idx)} venue={row.get('venue_name', '?')} outcome={row['outcome_class']} margin={row.get('projected_distance_to_wall_margin', float('nan')):.1f}ft (L1 change={l1_diff.loc[idx]:.3f}) ---")
        cdf = pd.DataFrame({
            "baseline_v02": proba_by_variant[VARIANT_BASELINE_V02].loc[idx],
            "venue_plus_geometry_v04_candidate": proba_by_variant[VARIANT_VENUE_PLUS_GEOMETRY_V04_CANDIDATE].loc[idx],
        })
        cdf["delta"] = cdf["venue_plus_geometry_v04_candidate"] - cdf["baseline_v02"]
        display(cdf)
else:
    print("Skipped -- no comparison available.")


--- event_id=745257-26-1 venue=T-Mobile Park outcome=home_run margin=34.2ft (L1 change=1.901) ---


,baseline_v02,venue_plus_geometry_v04_candidate,delta
out,0.376877,0.000153,-0.376724
single,0.263119,0.000862,-0.262257
double,0.239997,0.010420,-0.229577
triple,0.082785,0.000742,-0.082042
home_run,0.037222,0.987823,0.950600



--- event_id=745285-20-7 venue=Oracle Park outcome=home_run margin=25.1ft (L1 change=1.899) ---


,baseline_v02,venue_plus_geometry_v04_candidate,delta
out,0.852721,0.004027,-0.848693
single,0.032452,0.001497,-0.030955
double,0.057127,0.014741,-0.042385
triple,0.030245,0.002816,-0.027429
home_run,0.027456,0.976919,0.949462



--- event_id=745720-34-7 venue=Yankee Stadium outcome=home_run margin=22.3ft (L1 change=1.883) ---


,baseline_v02,venue_plus_geometry_v04_candidate,delta
out,0.796858,0.002480,-0.794379
single,0.040602,0.000545,-0.040057
double,0.115191,0.011355,-0.103836
triple,0.003353,0.000077,-0.003276
home_run,0.043996,0.985543,0.941547



--- event_id=746908-45-8 venue=Fenway Park outcome=out margin=41.1ft (L1 change=1.870) ---


,baseline_v02,venue_plus_geometry_v04_candidate,delta
out,0.859442,0.002745,-0.856698
single,0.030459,0.003027,-0.027432
double,0.054804,0.031841,-0.022963
triple,0.031312,0.003366,-0.027946
home_run,0.023983,0.959021,0.935038



--- event_id=745341-24-6 venue=Oracle Park outcome=home_run margin=25.6ft (L1 change=1.856) ---


,baseline_v02,venue_plus_geometry_v04_candidate,delta
out,0.809983,0.001765,-0.808218
single,0.037832,0.000407,-0.037424
double,0.061918,0.007068,-0.054850
triple,0.028613,0.001227,-0.027386
home_run,0.061654,0.989533,0.927879


## 15. Physically plausible interpretation

Read the example plays above together with `projected_distance_to_wall_margin` (positive = the ball's measured distance exceeded the wall distance along that spray direction). A physically plausible geometry effect looks like: rows where the margin is strongly positive should see probability mass shift TOWARD `home_run` (and away from `out`/fly-ball-caught outcomes) relative to the baseline, which does not know whether a given raw distance is "enough" at that specific park and direction -- and rows with a strongly negative margin (well short of the wall) should see the opposite. Compare this against Version 0.3's qualitative finding (`05_park_aware_analysis.ipynb`): `venue_id` alone already shifted Coors Field fly balls away from `home_run` toward `triple`/`double`, consistent with Coors' unusually deep fences. Geometry features give the model a DIRECT, park-specific "did this clear the fence here" signal instead of requiring it to infer that indirectly from venue identity -- if the shifts above track `projected_distance_to_wall_margin` in the expected direction, that is evidence the model is learning real park geometry, not just re-deriving venue identity through a different feature. This check is qualitative, not automated -- see the adoption recommendation below for why criterion 8 (physical plausibility) is never auto-passed.

## 16. Adoption recommendation

In [17]:
if comparison is not None and bootstrap_by_candidate is not None:
    for candidate in GEOMETRY_CANDIDATE_VARIANTS:
        comparison[candidate]["geometry_coverage_rate"] = float(val_df["has_park_geometry"].mean())
    recommendation = recommend_geometry_adoption(comparison, bootstrap_by_candidate)
    import json
    print(json.dumps(recommendation, indent=2, default=str))
else:
    recommendation = None
    print("Skipped -- no comparison available.")

{
  "per_candidate": {
    "geometry_only_v04_candidate": {
      "improves_log_loss": true,
      "log_loss_delta": -0.10703849711073998,
      "bootstrap_supports_improvement_or_no_meaningful_regression": true,
      "log_loss_bootstrap": {
        "metric": "log_loss",
        "point_estimate": -0.10703849711073998,
        "ci_low": -0.1094019169563889,
        "ci_high": -0.1041998283205662,
        "confidence_level": 0.95,
        "n_reps": 500,
        "seed": 42,
        "resampling_unit": "game_pk"
      },
      "ece_not_materially_worse": true,
      "ece_delta": 0.0018707914930718925,
      "home_run_ece_not_materially_worse": true,
      "home_run_ece_delta": -0.0016868677472557313,
      "no_material_venue_regressions": false,
      "material_venue_regressions": [
        "4169",
        "5340"
      ],
      "no_material_subgroup_regressions": true,
      "material_subgroup_regressions": [],
      "sufficient_geometry_coverage": true,
      "passes_all_automated_criteri

## 17. Explicit limitations

- **Geometry sourcing is agent-researched, not human-reviewed.** Every record in `mlb_luck_score.data.park_geometry.PARK_GEOMETRY_POINTS` has `review_status="agent_sourced_pending_human_review"` -- gathered from cited public sources (mostly Wikipedia) by an AI coding agent. A maintainer should spot-check records, especially any flagged in `notes` as simplified or uncertain, before treating this table as authoritative.
- **Five-point simplification.** Each park-era is modeled as five discrete wall points (left line, left-center, center, right-center, right line) with piecewise-linear interpolation between them -- real outfield walls curve continuously and have local irregularities (e.g. Citizens Bank Park's "Monty's Angle", Citi Field's former right-field "nook") that are not separately modeled.
- **Wall height is sparse.** Many reviewed points have no cited height at all (`wall_height_feet=None`) -- height-dependent features (`high_wall_indicator`, interaction terms) are `NaN`/imputed for those points, not guessed.
- **Temporary and neutral-site venues have NO geometry** (`TEMPORARY_OR_SPECIAL_VENUE_IDS`): Field of Dreams, the two 2021 Blue Jays alternate home venues, the London Series, the Mexico City Series, the Little League Classic field, and the 2024 Rickwood Field tribute game. These rows are retained in baseline analyses but excluded from geometry-specific subgroups.
- **No published park factors, weather, air density, or roof-state modeling.** Only static, season-independent wall geometry (except where a real mid-window renovation is captured as a second dated configuration).
- **`projected_distance_to_wall_margin` is not a home-run detector.** A positive margin means the measured `hit_distance_sc` exceeded the modeled wall distance along that spray direction -- it does NOT alone prove a ball cleared the fence (wall height, trajectory, spin, and Statcast measurement error are all unmodeled).
- **Contact Luck v0.4 remains a contact-only research prototype.** It does not yet model weather, air density, roof status, exact ball trajectory, defensive positioning, defensive execution, or batter-runner advancement.
- **2025 was never touched** by this analysis -- all training/validation/reference use only 2021-2023 (training) and 2024 (validation) data.

> **This repository is a research prototype, not a validated public baseball statistic.** See `README.md` and `CLAUDE.md` for full scope and limitations.